### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [11]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [12]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [13]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [14]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [15]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [16]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [17]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [18]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  6385,  1149, 11238])

Obtenemos los 5 documentos más similares:

In [19]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [20]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [21]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [22]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [23]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [24]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


### 1.
Tomo 5 documentos al azar y los comparo con los documentos mas similares:

In [33]:
np.random.seed(7)  # Semilla para reproducibilidad
random_indexes = np.random.choice(len(newsgroups_train.data), size=5, replace=False)

for idx in random_indexes:
  cossim = cosine_similarity(X_train[idx], X_train)[0]
  mostsim = np.argsort(cossim)[::-1][1:6] # Ordeno de mas a menos similar y tomo los 5 documentos mas cercanos

  print(f'Indice del documento: {idx}')
  print(f'Tag: {newsgroups_train.target_names[y_train[idx]]}')
  print('Contenido:')
  print(newsgroups_train.data[idx])
  print('Documentos mas cercanos:')

  for similar_idx in mostsim:
    print(f'Indice: {similar_idx}')
    print(f'Tag: {newsgroups_train.target_names[y_train[similar_idx]]}')
    print(newsgroups_train.data[similar_idx])
    print('-----------------------------------------')

  print('======================================')

Indice del documento: 5206
Tag: rec.autos
Contenido:



Shades of the Edsel! They had pushbuttons in the steering wheel hub
that controlled the auto tranny. It was very disconcerting to shift
into reverse when turning a corner and the wires shorted.






Documentos mas cercanos:
Indice: 3860
Tag: rec.autos

There is just something disconcerting about the name of this group. :)


-----------------------------------------
Indice: 5629
Tag: rec.autos

Well Sweden and Australia, and lord knows wherever else used to drive on
the "wrong" side of the road, so the export market might have been
larger then than just the UK.


By the 1920s, there was a very active "good roads" movement, which had
its origins actually in the 1890s during the bicycle craze, picked up
steam in the teens (witness the Linclon Highway Association, 1912 or so,
and the US highway support act (real name: something different) in 1916
that first pledged federal aid to states and counties to build decent
roads. Also, the e

Se observa que el primer documento es de categoría `rec.autos` y 3/5 de los documentos más similares según la similitud coseno tienen la misma categoría, uno de los que no coincide es de categoría hardware pero su contenido es simplemente `"auto from` y el último también es de categoría hardware pero su contenido sí tiene que ver con componentes de hardware.

El segundo texto es de categoría `comp.windows.x` que es un tag bastante específico y si bien ninguno de los documentos pertenece a esa misma categoría exactamente, todos pertenecen a la categoría más general `comp`.

El tercer documento es de categoría `rec.sport.hockey` pero su contenido no parece tener mucho que ver con el hockey, aún así 2 de los documentos más cercanos son de la misma categoría, otro es de un deporte diferente `rec.sport.baseball` y los restantes son de categoría `soc.religion.christian` y `talk.politics.mideast`, este último parece tratarse sobre racismo al igual que el documento original.

El cuarto documentos es de categoría `talk.politics.mideast` y los 5 documentos más similares encontrados comparten la misma categoría.

El último documento tiene categoría `sci.crypt`, aunque es bastante vago en su contenido: `As long as "you are on your own" means that you can use your own encryption, I'm sold.`, solamente por la palabra "encryption" nos podemos dar cuenta de la categoría. Solamente uno de los documentos encontrados pertenece a la misma categoría.

Se puede concluir que este es un método muy sencillo, rápido y que no utiliza muchos recursos y que funciona bastante bien si lo que se busca es claficar documentos en categorías generales.

### 2.
Clasificador por prototipos: asignamos a cada documento de test la clase del documento de entrenamiento más similar según coseno

In [ ]:
cossim_test_train = cosine_similarity(X_test, X_train)
best_train_idx = np.argmax(cossim_test_train, axis=1)
y_pred_proto = y_train[best_train_idx]

print('Prediccion para todos los documentos de test:')
for i in range(len(newsgroups_test.data)):
  print(f'Documento {i}: predicho: {newsgroups_train.target_names[y_pred_proto[i]]} | real: {newsgroups_test.target_names[y_test[i]]}')

print()
print(f"F1-score macro: {f1_score(y_test, y_pred_proto, average='macro'):.4f}")

Prediccion para todos los documentos de test:
Documento 0: predicho: alt.atheism | real: rec.autos
Documento 1: predicho: talk.religion.misc | real: comp.windows.x
Documento 2: predicho: talk.politics.mideast | real: alt.atheism
Documento 3: predicho: talk.politics.mideast | real: talk.politics.mideast
Documento 4: predicho: alt.atheism | real: talk.religion.misc
Documento 5: predicho: sci.med | real: sci.med
Documento 6: predicho: talk.politics.guns | real: soc.religion.christian
Documento 7: predicho: misc.forsale | real: soc.religion.christian
Documento 8: predicho: comp.windows.x | real: comp.windows.x
Documento 9: predicho: comp.graphics | real: comp.graphics
Documento 10: predicho: comp.os.ms-windows.misc | real: comp.os.ms-windows.misc
Documento 11: predicho: comp.graphics | real: comp.windows.x
Documento 12: predicho: talk.politics.mideast | real: talk.politics.mideast
Documento 13: predicho: rec.autos | real: rec.motorcycles
Documento 14: predicho: alt.atheism | real: alt.athe

El método obtuvo un F1 score de 0.505 que no está mal para ser un método tan sencillo y rápido. Como vimos en el punto anterior si bien a veces no se encontraban documentos con el mismo tag exacto pero sí con la categoría más general, hago una prueba a ver cuanto mejora el score si solo considero la primera parte del tag:

In [ ]:
train_top_labels = np.array([name.split('.')[0] for name in newsgroups_train.target_names])
test_top_labels = np.array([name.split('.')[0] for name in newsgroups_test.target_names])

y_train_top = train_top_labels[y_train]
y_test_top = test_top_labels[y_test]
y_pred_proto_top = train_top_labels[y_pred_proto]

print('Prediccion para todos los documentos de test usando solo la primera parte del tag:')
for i in range(len(newsgroups_test.data)):
  print(f'Documento {i}: predicho: {y_pred_proto_top[i]} | real: {y_test_top[i]}')

print()
print(f"F1-score macro: {f1_score(y_test_top, y_pred_proto_top, average='macro'):.4f}")

Prediccion para todos los documentos de test usando solo la primera parte del tag:
Documento 0: predicho: alt | real: rec
Documento 1: predicho: talk | real: comp
Documento 2: predicho: talk | real: alt
Documento 3: predicho: talk | real: talk
Documento 4: predicho: alt | real: talk
Documento 5: predicho: sci | real: sci
Documento 6: predicho: talk | real: soc
Documento 7: predicho: misc | real: soc
Documento 8: predicho: comp | real: comp
Documento 9: predicho: comp | real: comp
Documento 10: predicho: comp | real: comp
Documento 11: predicho: comp | real: comp
Documento 12: predicho: talk | real: talk
Documento 13: predicho: rec | real: rec
Documento 14: predicho: alt | real: alt
Documento 15: predicho: sci | real: comp
Documento 16: predicho: comp | real: comp
Documento 17: predicho: comp | real: comp
Documento 18: predicho: talk | real: misc
Documento 19: predicho: talk | real: talk
Documento 20: predicho: talk | real: comp
Documento 21: predicho: misc | real: misc
Documento 22: pr

Se obtuvo un F1 score de 0.592 que nuevamente no está mal al tratarse de un método muy básico.

### 3.
Naïve Bayes:

In [44]:
models = {
  'MultinomialNB': MultinomialNB(),
  'ComplementNB': ComplementNB(),
}

config_vectorizer = [
  ('base', TfidfVectorizer()),
  ('stop_words_english', TfidfVectorizer(stop_words='english')),
  ('min_df_5_max_df_0.5', TfidfVectorizer(min_df=5, max_df=0.8)),
  ('sublinear_tf', TfidfVectorizer(sublinear_tf=True)),
  ('user_idf_smooth_idf_false', TfidfVectorizer(use_idf=False, smooth_idf=False)),
]

results = []

for name, vectorizer in config_vectorizer:
  X_train_cfg = vectorizer.fit_transform(newsgroups_train.data)
  X_test_cfg = vectorizer.transform(newsgroups_test.data)

  for nombre_modelo, modelo in models.items():
    modelo.fit(X_train_cfg, y_train)
    y_pred_cfg = modelo.predict(X_test_cfg)
    score = f1_score(y_test, y_pred_cfg, average='macro')
    results.append((score, name, nombre_modelo))
    print(f'Vectorizer: {name} | Modelo: {nombre_modelo} | F1 score: {score:.4f}')

Vectorizer: base | Modelo: MultinomialNB | F1 score: 0.5854
Vectorizer: base | Modelo: ComplementNB | F1 score: 0.6930
Vectorizer: stop_words_english | Modelo: MultinomialNB | F1 score: 0.6468
Vectorizer: stop_words_english | Modelo: ComplementNB | F1 score: 0.6936
Vectorizer: min_df_5_max_df_0.5 | Modelo: MultinomialNB | F1 score: 0.6132
Vectorizer: min_df_5_max_df_0.5 | Modelo: ComplementNB | F1 score: 0.6838
Vectorizer: sublinear_tf | Modelo: MultinomialNB | F1 score: 0.5860
Vectorizer: sublinear_tf | Modelo: ComplementNB | F1 score: 0.6920
Vectorizer: user_idf_smooth_idf_false | Modelo: MultinomialNB | F1 score: 0.4716
Vectorizer: user_idf_smooth_idf_false | Modelo: ComplementNB | F1 score: 0.6677


Se probaron algunos parámetros del vectorizador:

- `stop_words='english'`: elimina las stopwords que son palabras muy comunes que no aportan significado.
- `min_df=5`: descarta palabras que aparecen en menos de 5 documentos por considerarse errores o palabras demasiado específicas, por default es 1.
- `max_df=0.8`: descarta palabras que aparecen en el 80% o más de los documentos por considerarse muy comunes, por default es 1.0.
- `sublinear_tf=True`: suaviza el peso de los términos muy repetidos para que no dominen tanto.
- `use_idf=False`: desactiva el peso IDF y deja solo el conteo de términos.
- `smooth_idf=False`: desactiva el suavizado del IDF.

Se observa en todas las variantes probadas `ComplementNB` funcionó mejor que `MultinomialNB` para este caso. Las variaciones en algunos parámetros solo lograron mejorar significativamente para `MultinomialNB`, en cambio para `ComplementNB` solo se consiguió una mejora muy pequeña agregando las stop words.


### 4.
Encuentro palabras similares mediante la matriz término-documento:

In [54]:
X_train_td = X_train.T
palabras = ['sport', 'car', 'internet', 'government', 'religion']

for palabra in palabras:
  idx_palabra = tfidfvect.vocabulary_[palabra]
  cossim_palabras = cosine_similarity(X_train_td[idx_palabra], X_train_td)[0]
  mostsim_palabras = np.argsort(cossim_palabras)[::-1][1:6]

  print(f'Palabra: {palabra}')
  print('Palabras mas similares:')

  for similar_idx in mostsim_palabras:
    print(f'  - {idx2word[similar_idx]}')
  print('-----------------------------------------')

Palabra: sport
Palabras mas similares:
  - woops
  - pigeonhole
  - rec
  - golf
  - fj1200
-----------------------------------------
Palabra: car
Palabras mas similares:
  - cars
  - criterium
  - civic
  - owner
  - dealer
-----------------------------------------
Palabra: internet
Palabras mas similares:
  - constrained
  - nren
  - businesses
  - innate
  - superhighway
-----------------------------------------
Palabra: government
Palabras mas similares:
  - the
  - to
  - of
  - libertarian
  - encryption
-----------------------------------------
Palabra: religion
Palabras mas similares:
  - religious
  - religions
  - categorized
  - purpsoe
  - crusades
-----------------------------------------


Se observa que si bien algunas de las palabras encontradas sí están relacionadas, la mayoría no parecen tener mucha relación (ejemplo internet - innate) o son demasiado específicas (ejemplo sport - fj1200). También encontró algunas stopwords (ejemplo government - the, to, of) y variaciones de una misma palabra (ejemplo car - cars) al no estar utilizando técnicas como remover stopwords y de stemming o lemmatization.

Podemos concluir que si bien logra capturar algunos conceptos y categorías generales, este método no logra capturar muy bien el significado de palabras individuales.